In [6]:

# =========================
# 📦 Imports
# =========================
import pandas as pd
import requests
import time
from google.colab import files

# =========================
# 📥 LOAD FILES (IMPORTANT FIX)
# =========================

# Flights file (لازم تغير الاسم حسب ملفك)
flights_df = pd.read_csv("/content/aviationstack_flights_clean.csv")

# Airports file
airports_df = pd.read_csv("/content/world-airports.csv")

# =========================
# 🧠 SAFE CHECK (prevents NameError)
# =========================
if "Departure IATA" not in flights_df.columns:
    raise Exception("❌ Flights file missing 'Departure IATA' column")

if "Arrival IATA" not in flights_df.columns:
    raise Exception("❌ Flights file missing 'Arrival IATA' column")

# =========================
# ✈️ Extract used airports only
# =========================
used_airports = set()

used_airports.update(flights_df["Departure IATA"].dropna().unique())
used_airports.update(flights_df["Arrival IATA"].dropna().unique())

airports_clean = airports_df[
    airports_df["iata_code"].isin(used_airports)
].dropna(subset=["latitude_deg", "longitude_deg"]).copy()

print("📍 Airports used:", len(airports_clean))

# =========================
# 🌦️ Weather collection
# =========================
weather_rows = []

start_date = "2023-01-01"
end_date = "2023-01-10"

def get_weather(lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "wind_speed_10m_max"
        ],
        "timezone": "auto"
    }

    try:
        r = requests.get(url, params=params, timeout=15)
        return r.json()
    except:
        return None

# =========================
# 🚀 MAIN LOOP
# =========================
print("🌦️ Collecting weather data...")

for i, row in airports_clean.iterrows():

    lat = row["latitude_deg"]
    lon = row["longitude_deg"]
    iata = row["iata_code"]

    data = get_weather(lat, lon)

    if data and "daily" in data:

        dates = data["daily"]["time"]

        for j in range(len(dates)):
            weather_rows.append({
                "iata_code": iata,
                "date": dates[j],
                "temp_max": data["daily"]["temperature_2m_max"][j],
                "temp_min": data["daily"]["temperature_2m_min"][j],
                "precipitation": data["daily"]["precipitation_sum"][j],
                "wind_speed": data["daily"]["wind_speed_10m_max"][j]
            })

    print(f"✔ Done: {iata}")

    time.sleep(0.1)

# =========================
# 📊 FINAL DATAFRAME
# =========================
weather_df = pd.DataFrame(weather_rows)

print("\n✅ Weather dataset ready:", weather_df.shape)
display(weather_df.head())

# =========================
# 💾 SAVE + DOWNLOAD
# =========================
file_name = "weather_dataset_final.csv"
weather_df.to_csv(file_name, index=False)

print("💾 Saved successfully!")

files.download(file_name)

print("DONE ✅")

📍 Airports used: 386
🌦️ Collecting weather data...
✔ Done: LHR
✔ Done: LAX
✔ Done: ORD
✔ Done: JFK
✔ Done: ATL
✔ Done: CDG
✔ Done: SFO
✔ Done: AMS
✔ Done: FRA
✔ Done: DFW
✔ Done: EWR
✔ Done: DEN
✔ Done: MIA
✔ Done: IAD
✔ Done: SEA
✔ Done: BOS
✔ Done: LGW
✔ Done: IAH
✔ Done: DTW
✔ Done: YYZ
✔ Done: MUC
✔ Done: BCN
✔ Done: FCO
✔ Done: MAD
✔ Done: DUB
✔ Done: CPH
✔ Done: ZRH
✔ Done: BRU
✔ Done: STN
✔ Done: DXB
✔ Done: VIE
✔ Done: YVR
✔ Done: SIN
✔ Done: NRT
✔ Done: LIS
✔ Done: YUL
✔ Done: ARN
✔ Done: HKG
✔ Done: PRG
✔ Done: SYD
✔ Done: BKK
✔ Done: MXP
✔ Done: BUD
✔ Done: ATH
✔ Done: OSL
✔ Done: MAN
✔ Done: ISL
✔ Done: HEL
✔ Done: EDI
✔ Done: PEK
✔ Done: GVA
✔ Done: ORY
✔ Done: KUL
✔ Done: DUS
✔ Done: LTN
✔ Done: VCE
✔ Done: WAW
✔ Done: MEX
✔ Done: ICN
✔ Done: PMI
✔ Done: AGP
✔ Done: NCE
✔ Done: DEL
✔ Done: PVG
✔ Done: MEL
✔ Done: HND
✔ Done: JNB
✔ Done: HAM
✔ Done: GLA
✔ Done: GRU
✔ Done: TLV
✔ Done: CGN
✔ Done: SVO
✔ Done: AKL
✔ Done: DOH
✔ Done: TPE
✔ Done: CAI
✔ Done: AUH
✔ Done: MLA
✔

,iata_code,date,temp_max,temp_min,precipitation,wind_speed
0,LHR,2023-01-01,11.8,8.4,5.5,30.9
1,LHR,2023-01-02,8.4,0.3,1.4,12.5
2,LHR,2023-01-03,12.4,0.8,4.0,33.5
3,LHR,2023-01-04,12.9,10.1,1.1,31.6
4,LHR,2023-01-05,12.9,7.8,0.0,26.9


💾 Saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DONE ✅
